In [ ]:
import os
import pandas as pd
import openai
from tqdm import tqdm
import ast

# -----------------------------
# Setup paths
# -----------------------------
path_wd = "D:/Projektfolder1 (Miethe, Grosenick)/zz_AmelieMisc/"
path_data = os.path.join(path_wd, "thankyou/data/")

os.chdir(path_wd)


# -----------------------------
# OpenAI API key
# -----------------------------

# set key
openai.api_key = ""

In [45]:

# -----------------------------
# Load data (RData converted to CSV for Python)
# -----------------------------
# If you have a CSV export from RData:
wos_data_ai = pd.read_csv(os.path.join(path_data, "gen/wos_data_for_ai_missing.csv"))

# For testing, limit to first 10 rows
# wos_data_ai = wos_data_ai.head(10)
wos_data_ai.shape



(1464, 8)

In [46]:

# -----------------------------
# AI extraction function
# -----------------------------
def extract_names_ai(text: str):
    prompt = f"""
Extract only the names of people thanked for comments, suggestions, or feedback.
If someone is mentioned as editor or co-editor, write that down. For this, create a structure with two columns: 'person_name' and 'type'.
Also mention if someone is named as just edit or co-editor without name. Name is then NA.
Ignore mentions of research assistance, data access, seminar participants, or generic terms like 'all participants'.
Output the result as a Python list of dictionaries, e.g.:
[{{'person_name': 'John Doe', 'type': 'comment'}}, {{'person_name': None, 'type': 'co-editor'}}]

Text: "{text}"
"""
    try:
        response = openai.chat.completions.create(
            model="gpt-5-mini", # before gpt-5-mini
            messages=[{"role": "user", "content": prompt}]
        )
        result_text = response.choices[0].message.content
        
        try:
            names_list = ast.literal_eval(result_text)
        except:
            names_list = []
            
        return names_list
    except Exception as e:
        print(f"AI call failed: {e}")
        return []


In [47]:

# -----------------------------
# Loop over funding_text column
# -----------------------------
rows = []

for idx, row in tqdm(wos_data_ai.iterrows(), total=wos_data_ai.shape[0], desc="AI extracting"):
    file_name = row['title_paper']
    funding_text = row['funding_text']
    
    extracted = extract_names_ai(funding_text)
    
    for entry in extracted:
        person_name = entry.get('person_name', None)
        typ = entry.get('type', None)
        rows.append({'file_name': file_name, 'person_name': person_name, 'type': typ})

# -----------------------------
# Combine into a DataFrame
# -----------------------------
df_names = pd.DataFrame(rows)

# -----------------------------
# Save results
# -----------------------------
output_file = os.path.join(path_data, "gen/ai_extracted_names_wos_other_journals_3.csv")
df_names.to_csv(output_file, index=False)

print(df_names)

AI extracting: 100%|██████████| 1464/1464 [1:59:18<00:00,  4.89s/it]  

                                             file_name     person_name  \
0    Moving to Job Opportunities? The Effect of "Ba...     Amanda Agan   
1    Moving to Job Opportunities? The Effect of "Ba...  Steven Raphael   
2    Moving to Job Opportunities? The Effect of "Ba...  Sarit Weisburd   
3    Leadership in Social Movements: Evidence from ...    Esther Duflo   
4    Leadership in Social Movements: Evidence from ...  Daron Acemoglu   
..                                                 ...             ...   
476  Does "being chosen to lead" induce non-selfish...   Erik Snowberg   
477  Does "being chosen to lead" induce non-selfish...     Leeat Yariv   
478  Does "being chosen to lead" induce non-selfish...            None   
479  Subsidizing low- and middle-income adoption of...        Tom Knox   
480  Subsidizing low- and middle-income adoption of...        Mei Wang   

          type  
0      comment  
1      comment  
2      comment  
3    co-editor  
4      comment  
..       

In [ ]:

# -----------------------------
# Combine into a DataFrame
# -----------------------------
df_names = pd.DataFrame(rows)

# -----------------------------
# Save results
# -----------------------------
output_file = os.path.join(path_data, "gen/ai_extracted_names_wos_other_journals_2.csv")
df_names.to_csv(output_file, index=False)
